In [ ]:
pip install ragas==0.2.6 langchain-google-genai datasets --break-system-packages

In [ ]:
import os, pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

GEMINI_KEY = os.environ.get("GEMINI_API_KEY")
print(f"Gemini: {'SET' if GEMINI_KEY else 'NOT SET'}")

# ── Load saved answers ────────────────────────────────────────────────────
# Change this path to wherever you extracted the zip
eval_df = pd.read_parquet(r"D:\SciRet-Scientific-Information-Made-Easy\Sciret2\4_results\tier2\eval_runs_tier2.parquet")
print(f"Loaded {len(eval_df)} eval runs")
print(eval_df["system"].value_counts())

# ── RAGAS with Gemini ─────────────────────────────────────────────────────
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=GEMINI_KEY,
        convert_system_message_to_human=True
    )
)
ragas_emb = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(
        model="models/embedding-001",
        google_api_key=GEMINI_KEY
    )
)

ragas_results = {}
for sys_name in ["dpr_baseline", "sciret_text", "sciret_multi"]:
    subset = eval_df[eval_df["system"] == sys_name]
    ds = Dataset.from_dict({
        "question":     subset["question"].tolist(),
        "answer":       subset["answer"].tolist(),
        "contexts":     subset["contexts"].tolist(),
        "ground_truth": subset["ground_truth"].tolist(),
    })
    try:
        result = evaluate(ds,
            metrics=[faithfulness, answer_relevancy,
                     context_precision, context_recall],
            llm=ragas_llm,
            embeddings=ragas_emb
        )
        ragas_results[sys_name] = dict(result)
        print(f"\n{sys_name}:")
        for k, v in dict(result).items():
            print(f"  {k}: {v:.3f}")
    except Exception as e:
        print(f"{sys_name} failed: {e}")

ragas_df = pd.DataFrame.from_dict(ragas_results, orient="index")
print("\n=== FINAL RAGAS RESULTS ===")
print(ragas_df.to_string())
ragas_df.to_csv("ragas_tier2_fixed.csv")
print("\nSaved to ragas_tier2_fixed.csv")